In [568]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 数据预处理

In [569]:
# l: communication rate level,  action -> communication mode
# r: road condition
# m: The acknowledgement message satisfies the condition m<ack> \in {1, 2}, where 1 means ACK success, and 2 means retransmission required.
# X_t: The accident indicator satisfies the condition X_t \in {0, 1}, where 0 denotes that no accident occurs at time t, and 1 indicates that an unexpected hazardous event happens.
# p_u: The unexpected-event probability p_u denotes the likelihood of an accident occurring at time t. A larger p_u indicates a higher risk of unexpected hazardous events. (state)
# p_k: The radar miss probability p_k represents the miss-detection rate of radar mode k under the current traffic condition. A lower p_k implies better radar detection performance. (state)
# delta_i: The message age delta_i denotes the waiting time of the i-th DENM message in the queue, measured as the duration between its arrival and transmission. Larger delta_i indicates staler emergency information. (state)
# n: The traffic-density indicator n describes the surrounding-vehicle condition, reflecting how congested the road environment is. Higher n corresponds to denser nearby traffic. (state)
# weather: The weather indicator represents the current weather condition on a 1–10 hazard scale, where larger values correlate with a higher risk of accidents due to adverse weather factors. (state)
# v: vehicle speed

state = {
    "weather"
    "r",
    "v",
    "m",
    "n",
}

# X_t, p_u, p_k, delta_i
    

## 环境变量向量

$$
\begin{align*}

e &:\ [w, r, n, v, m^{\text{<ack>}}] \\
e &:\ \text{environmental variable vector} \\
w &:\ \text{weather condition}  \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
r &:\ \text{road condition}  \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
n &:\ \text{surrounding vehicles condition} \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
v &:\ \text{vehicle speed} \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \}

\end{align*}
$$


$$
\text{hazard level } h = \begin{cases}
    1 & \text{lowest accident risk} \\
    10 & \text{highest accident risk}
\end{cases}
$$


$$
\text{ACK message } m^{\text{<ack>}} = \begin{cases}
    1 & \text{ACK received (no retransmission)} \\
    2 & \text{ACK not received (retransmission required)}
\end{cases}
$$

### 正态分布
$$
\begin{align*}

\mathcal{X} \sim \mathcal{N}(\mu, \sigma^{2})

\end{align*}
$$

#### 密度函数
$$
\begin{align*}

\mathcal{f(x)} = \frac{1}{\sigma \sqrt{2 \pi}} \exp{
    \left(
        -\frac{(x - \mu)^{2}}{2 \sigma^{2}}
    \right)
}

\end{align*}
$$

$$
\begin{align*}

w = \mathrm{Clamp}_{[1, 10]} \left(
        \mathrm{Round} \left(
            \mathcal{N}(5.5, 2^2)
        \right)
    \right)

\end{align*}
$$

In [570]:
def generate_weather_condition(batch_size, device = "cpu"):
    w = torch.normal(5.5, 2.0, size = (batch_size, ), device = device)
    return torch.clamp(w.round(), 1, 10)

$$
\begin{align*}

r_{\mathrm{prior}} &= \mathcal{N}(5.5, 1^{2}) \\

r &= \mathrm{Clamp}_{[1, 10]} \left(
    \mathrm{Round} \left(
        r_{\mathrm{prior}} + 0.5(w - 5.5) + \mathcal{N}(0, 1^{2})
    \right)
\right)

\end{align*}
$$

In [571]:
def generate_road_condition(w, device = "cpu"):
    batch_size = w.shape[0]
    prior = torch.normal(5.5, 1.0, size = (batch_size, ), device = device)
    noise = torch.normal(0, 1.0, size = (batch_size, ), device = device)
    r = prior + 0.5 * (w - 5.5) + noise
    return torch.clamp(r.round(), 1, 10)

$$
\begin{align*}

d_{\mathrm{pre}} &= r_{min} + r_{\max} - r = 11 - r \\

d &= \mathrm{Clamp}_{[1, 10]} \left( d_{\mathrm{pre}} + \mathcal{N}(0, 1^{2}) \right)

\end{align*}
$$


In [572]:
def generate_vehicle_density(r, device = "cpu"):
    batch_size = r.shape[0]
    d_pre = 11.0 - r
    noise = torch.normal(0, 1.0, size = (batch_size, ), device = device)
    d = d_pre + noise
    return torch.clamp(d, 1, 10)

### 高斯函数
$$
\begin{align*}

\mathcal{f(x)} = A \exp{
    \left(
        -\frac{(x - \mu)^{2}}{2\sigma^{2}}
    \right)
}

\end{align*}
$$

$$
\begin{align*}

n_{\mathrm{pre}} &= A \exp{
    \left(
        -\frac{(d - \mu)^{2}}{2 \sigma^{2}}
    \right)
} \\


 &= 10 \cdot \exp \left( 
    -\frac{(d - 5.5)^{2}}{2 \cdot 2^{2}}
\right)

\end{align*}
$$


$$
n = \mathrm{Clamp}_{[1, 10]} \left(
    \mathrm{Round} \left(
        n_{\mathrm{pre}} + \mathcal{N}(0, 1^{2})
    \right)
\right)
$$


In [573]:
def generate_surrounding_vehicles_condition(d, device = "cpu"):
    batch_size = d.shape[0]
    n_pre = 10.0 * torch.exp(- ((d - 5.5) ** 2) / (2 * (2.0 ** 2)))
    noise = torch.normal(0, 1.0, size = (batch_size, ), device = device)
    n = n_pre + noise
    return torch.clamp(n.round(), 1, 10)

$$
\begin{align*}

V \sim 
\begin{cases}
\mathcal{N}(0, 1) & &u < 0.70 \\
\mathcal{N}(0, 3) & 0.70 \le &u < 0.95 \\
\mathcal{N}(0, 6) & &u \ge 0.95
\end{cases}

\quad u &\sim U(0,1)

\quad v_{\text{prior}} &= |V|

\end{align*} \\


\begin{align*}

v = \mathrm{Clamp}_{[1, 10]} \left(
    \mathrm{Round} \left(
        v_{\mathrm{prior}} + 0.5(d - 5.5) + \mathcal{N}(0, 1^{2})
    \right)
\right)

\end{align*} 
$$

In [574]:
def generate_v_prior(batch_size, device = "cpu"):
    u = torch.rand(batch_size, device = device)
    mask1 = u < 0.70
    mask2 = (u >= 0.70) & (u < 0.95)
    mask3 = u >= 0.95

    v = torch.empty(batch_size, device = device)
    v[mask1] = torch.normal(0, 1.0, size = (mask1.sum(), ), device = device)
    v[mask2] = torch.normal(0, 3.0, size = (mask2.sum(), ), device = device)
    v[mask3] = torch.normal(0, 6.0, size = (mask3.sum(), ), device = device)

    return v.abs()

In [575]:
def generate_vehicle_speed(d, device = "cpu"):
    batch_size = d.shape[0]
    prior = generate_v_prior(batch_size, device = device)
    noise = torch.normal(0, 1.0, size = (batch_size, ), device = device)
    v = prior + 0.5 * (d - 5.5) + noise
    return torch.clamp(v.round(), 1, 10)

### received signal to interference plus noise ratio(SINR)
$$
\begin{align*}

\eta = \frac{P_{R}}{\sigma_{I} + \sigma_{N}}

\end{align*}
$$

$$
\begin{align*}

P_{R} &= \alpha(w_{\max} + w_{\min} - w) + \beta(d_{\max} + d_{\min} - d) + \mathcal{N}(0, 1^{2}) \\

&= \alpha(11 - w) + \beta(11 - d) + \mathcal{N}(0, 1^{2}) \\

&= 1.2(11 - w) + 0.8(11 - d) + \mathcal{N}(0, 1^{2}) \\
\\
\sigma_{I} &= \beta_{1}d + \mathcal{N}(0, 1) = 0.4d + \mathcal{N}(0, 1) \\
\sigma_{N} &= \sigma_{0} + \mathcal{N}(0, 0.1) = 1 + \mathcal{N}(0, 0.1) \\

\end{align*}
$$

In [576]:
def generate_SINR(w, d, device = "cpu"):
    batch_size = w.shape[0]

    PR = (1.2 * (11 - w) + 0.8 * (11 - d) + torch.normal(0, 1.0, size=(batch_size,), device=device)).relu()

    sigma_I = (0.4 * d + torch.normal(0, 1.0, size=(batch_size,), device=device)).relu()

    sigma_N = (1.0 + torch.normal(0, 0.1, size=(batch_size,), device=device)).relu()
    
    sinr = PR / (sigma_I + sigma_N + 1e-6)
    return sinr

In [577]:
def generate_channel_condition(sinr):
    """
    根据 SINR（线性值）生成 channel condition c ∈ {1,...,10}
    """
    # 1. 转 dB
    sinr_db = 10 * torch.log10(sinr + 1e-6)

    # 2. 量化区间：[-5 dB, 20 dB] → 10 档
    c = torch.floor((sinr_db + 5) / 2.5) + 1

    # 3. clamp 到 1~10
    c = torch.clamp(c, 1, 10)

    return c.long()


$$
\begin{align*}

l_{action} = l_{\text{optimal}} = 
\begin{cases}
1 & c \in \{ 1, 2 \} \\
2 & c \in \{ 3, 4, 5 \} \\
3 & c \in \{ 6, 7, 8 \} \\
4 & c \in \{ 9, 10 \}
\end{cases} 

\end{align*}
$$

In [578]:
def generate_communication_rate(c):
    l = torch.ones_like(c)
    l = torch.where((3 <= c) & (c <= 5), 2, l)
    l = torch.where((6 <= c) & (c <= 8), 3, l)
    l = torch.where(c >= 9, 4, l)
    return l

$$
\begin{align*}

m^{\text{<ack>}} = 
\begin{cases}
1 & l_{\text{action}} \leq l_{\text{optimal}} \\
2 & l_{\text{action}} > l_{\text{optimal}}
\end{cases}

\end{align*}
$$

In [579]:
def generate_m_ack(l_action, l_optimal):
    success = (l_action <= l_optimal).long()
    m_ack = torch.where(success == 1, torch.ones_like(success), torch.full_like(success, 2))
    return m_ack

In [580]:
def generate_envs(batch_size, device='cpu'):
    w = generate_weather_condition(batch_size, device)
    r = generate_road_condition(w, device)
    d = generate_vehicle_density(r, device)
    n = generate_surrounding_vehicles_condition(d, device)
    v = generate_vehicle_speed(d, device)
    m_ack = generate_m_ack(w, n, device)

    return torch.stack([w, r, n, v, m_ack], dim = 1)


## 状态

$$
\begin{align*}

s &:\ [e, c, q, m] \\
s &:\ \text{state} \\
e &:\ \text{environmental variable vector} \\
c &:\ \text{channel condition}  \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
q &:\ \text{status of the DENM message queue, indicates the age of the oldest DENM message in the current queue.} \\
m &:\ \text{message} \in \{ 1, 2, 3 \} \\

\end{align*}
$$

In [581]:
# old_c

### 消息队列

In [ ]:
class VehicleMessageQueue:
    def __init__(self, batch_size, q_max, device = "cpu"):
        self.batch_size = batch_size
        self.q_max = q_max
        self.device = device

        # CAM 队列：每个 world 是一个数值
        self.cam_count = torch.zeros(batch_size, dtype = torch.int32, device = device)
        self.denm_queues = [[] for _ in range(batch_size)]

    # ---------------------------------------------------
    # 1. 所有 DENM age += 1
    # ---------------------------------------------------
    def age_step(self):
        for i in range(self.batch_size):
            if len(self.denm_queues[i]) > 0:
                self.denm_queues[i] = [age + 1 for age in self.denm_queues[i]]

    # ---------------------------------------------------
    # 2. 丢弃超过 q_max 的 DENM（老化 + 过期）
    # ---------------------------------------------------
    def drop_expired(self):
        for i in range(self.batch_size):
            if len(self.denm_queues[i]) > 0:
                self.denm_queues[i] = [age for age in self.denm_queues[i] if age <= self.q_max]

    # ---------------------------------------------------
    # 3. 泊松到达事件（CAM + DENM）
    #    λ_cam、λ_denm 可以是标量或 batch_size 张量
    # ---------------------------------------------------
    def arrival_step(self, lambda_cam, lambda_denm):
        # 先处理 CAM
        cam_arrivals = np.random.poisson(lambda_cam)
        self.cam_count += torch.tensor(cam_arrivals, dtype=torch.int32, device=self.device)

        # 再处理 DENM
        denm_arrivals = np.random.poisson(lambda_denm)
        for i in range(self.batch_size):
            for _ in range(denm_arrivals[i]):
                self.denm_queues[i].append(0)

    # ---------------------------------------------------
    # 4. 根据 action 发送消息（CAM / DENM）
    # msg_type: 1=CAM, 2=DENM_normal, 3=DENM_accident
    # l_action: batch_size tensor, 每个 world 都有自己的发送速率
    # ---------------------------------------------------
    def transmit(self, msg_type, l_action):
        # msg_type 可以是 batch_size 的张量，也可以是一个统一的标量
        if isinstance(msg_type, int):
            msg_type = torch.full((self.batch_size,), msg_type, dtype=torch.int32, device=self.device)

        for i in range(self.batch_size):
            if msg_type[i] == 1:  
                # CAM
                sent = min(l_action[i].item(), self.cam_count[i].item())
                self.cam_count[i] -= sent

            else:
                # DENM
                queue_len = len(self.denm_queues[i])
                sent = min(l_action[i].item(), queue_len)
                self.denm_queues[i] = self.denm_queues[i][sent:]

    # ---------------------------------------------------
    # 5. 返回 batch_size 的 q 值（最老 DENM age）
    # ---------------------------------------------------
    def get_q(self):
        q_vals = []
        for i in range(self.batch_size):
            if len(self.denm_queues[i]) == 0:
                q_vals.append(0)
            else:
                q_vals.append(max(self.denm_queues[i]))
        return torch.tensor(q_vals, dtype=torch.float32, device=self.device)

    # ---------------------------------------------------
    # 6. 返回 batch_size 的 m 值（CAM=1, DENM_normal=2, DENM_accident=3）
    #    事故类型由上层环境决定，此处只区分 DENM vs CAM
    # ---------------------------------------------------
    def get_m(self, accident_mask=None):
        # accident_mask 是 batch_size 的 0/1 张量，用于区分普通 DENM vs 事故 DENM
        if accident_mask is None:
            accident_mask = torch.zeros(self.batch_size, dtype=torch.int32, device=self.device)

        m_vals = []
        for i in range(self.batch_size):
            if len(self.denm_queues[i]) == 0:
                m_vals.append(1)  # CAM
            else:
                if accident_mask[i].item() == 1:
                    m_vals.append(3)  # DENM involved accident
                else:
                    m_vals.append(2)  # DENM normal

        return torch.tensor(m_vals, dtype=torch.int32, device=self.device)

    # ---------------------------------------------------
    # 重置整个消息队列（可选）
    # ---------------------------------------------------
    def reset(self):
        self.cam_count.zero_()
        self.denm_queues = [[] for _ in range(self.batch_size)]


# ADMM-Net

$$
\begin{align*}

\textbf{X}&\textbf{-update (Low-rank Module)} \\

\mathcal{D}_{\tau}(X) &= \mathrm{Conv2D} \left( \mathrm{ReLU_{\tau} \left( \mathrm{Conv2D}(X) \right) } \right) \\

\mathrm{X}^{(k + 1)} &= \mathrm{Conv2D} \left( \mathrm{ReLU} \left( \mathrm{Conv2D} \left( Z^{(k)} - U^{(k)} \right) - \frac{\rho}{2} \right) \right)

\end{align*}
$$

In [583]:
class UpdateBlockX(nn.Module):
    def __init__(self, conv1, conv2, init_tau = 0.1):
        super().__init__()
        self.conv1 = conv1
        self.conv2 = conv2
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype = torch.float32))  # ρ/2

    # [B, 1, 1, F] -> [B, h, 1, F] -> [B, 1, 1, F]
    def forward(self, Z0_minus_U0):
        return self.conv2(F.relu(self.conv1(Z0_minus_U0) - self.tau))

$$
\begin{align*}

\textbf{Z}&\textbf{-update (Threshold Module)} \\
T &= X^{(k+1)} - U^{(k)} + Z^{(k)} \\
Z^{(k+1)} &= Soft_{\tau}(T) = sign(T) · \mathrm{ReLU} \left( |T| - \frac{\lambda}{\rho} \right) \\

\end{align*}
$$

In [584]:
class UpdateBlockZ(nn.Module):
    def __init__(self, init_tau = 0.1):
        super().__init__()
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype = torch.float32))  # λ/ρ

    def forward(self, X1, Z0_minus_U0):
        T = X1 + Z0_minus_U0
        
        return torch.sign(T) * F.relu(torch.abs(T) - self.tau)

$$
\begin{align*}

\textbf{U-update}&\textbf{ (Multiplier Module)} \\
U^{(k+1)} &= U^{(k)} + X^{(k+1)} - Z^{(k+1)} \\

\end{align*}
$$


In [585]:
class UpdateBlockU(nn.Module):
    def forward(self, U0, X1, Z1):
        return U0 + X1 - Z1

In [586]:
class AdmmBlock(nn.Module):
    def __init__(self, conv1, conv2, init_tau_x = 0.1, init_tau_z = 0.1):
        super().__init__()
        self.x_updater = UpdateBlockX(conv1, conv2, init_tau = init_tau_x)
        self.z_updater = UpdateBlockZ(init_tau = init_tau_z)
        self.u_updater = UpdateBlockU()

    def forward(self, Z0, U0):
        Z0_minus_U0 = Z0 - U0
        X1 = self.x_updater(Z0_minus_U0)
        Z1 = self.z_updater(X1, Z0_minus_U0)
        U1 = self.u_updater(U0, X1, Z1)
        
        return X1, Z1, U1

In [587]:
class AdmmNet(nn.Module):
    def __init__(self,iter_count = 6, conv_channel_cnt = 8, init_tau_x = 0.1, init_tau_z = 0.1):
        super().__init__()
        self.iter_count = iter_count

        self.shared_conv1 = nn.Conv2d(
            in_channels = 1,
            out_channels = conv_channel_cnt,
            kernel_size = 3,
            padding = 1,
            bias = True
        )

        self.shared_conv2 = nn.Conv2d(
            in_channels = conv_channel_cnt,
            out_channels = 1,
            kernel_size = 3,
            padding = 1,
            bias = True
        )

        blocks = []
        for _ in range(iter_count):
            blocks.append(
                AdmmBlock(
                    conv1 = self.shared_conv1,
                    conv2 = self.shared_conv2,
                    init_tau_x = init_tau_x,
                    init_tau_z = init_tau_z
                )
            )

        self.blocks = nn.ModuleList(blocks)

    def forward(self, Z0, U0):
        Z, U = Z0, U0
        X = None
        for block in self.blocks:
            X, Z, U = block(Z, U)

        return X, Z, U

In [588]:
def save_admm_net(model, path):
    torch.save(model.state_dict(), path)
    print(f"已保存 ADMM-Net 模型至: {path}")

def load_admm_net(config, path):
    iter_count = config["iter_count"]
    init_tau_x = config["init_tau_x"]
    init_tau_z = config["init_tau_z"]
    conv_channel_cnt = config["conv_channel_cnt"]
    
    model = AdmmNet(iter_count, conv_channel_cnt, init_tau_x, init_tau_z)
    state_dict = torch.load(path, map_location = torch.device("cpu"))
    model.load_state_dict(state_dict)
    
    print(f"已加载 ADMM-Net 模型自: {path}")
    
    return model

In [589]:
def calc_loss(
    mask: torch.Tensor,
    pred: torch.Tensor,
    tgt: torch.Tensor,
    z: torch.Tensor = None,
    delta: float = 1.0,
    lambda_nuc: float = 0.0
) -> tuple[torch.Tensor, int]:
    """
    Huber loss + 简化核范数（Z）正则项。
    Z 被视为一个 B×F 的矩阵，核范数定义为所有奇异值之和。
    """
    missing = ~mask
    diff = pred[missing] - tgt[missing]
    abs_diff = torch.abs(diff)

    huber = torch.where(
        abs_diff <= delta,
        0.5 * diff ** 2,
        delta * (abs_diff - 0.5 * delta)
    )

    loss = huber.mean()

    if z is not None and lambda_nuc > 0:
        # z: [B, 1, 1, F] -> [B, F]
        z_matrix = z.view(z.size(0), -1)  # [B, F]
        # 做一次整体 SVD，获取核范数
        s = torch.linalg.svdvals(z_matrix)  # 奇异值向量
        loss += lambda_nuc * s.sum()

    return loss, diff.numel()


In [590]:
from tqdm import tqdm

In [591]:
def train_admm_net(device, model, dataloader, optimizer, epoch, epoch_per_step, epoch_count, model_dir, model_name):
    model.train()

    for e in range(epoch, epoch_count + 1):
        total_loss = 0.0
        total_missing = 0

        progress = tqdm(dataloader, desc=f"epoch {e}/{epoch_count}")
        for x, m, y in progress:
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            p, z, _ = model(x, torch.zeros_like(x))

            loss, missing_count = calc_loss(mask = m, pred = p, tgt = y, z = z, lambda_nuc = 1e-4)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * missing_count
            total_missing += missing_count

            progress.set_postfix(loss=loss.item(), missing=missing_count)

        avg_loss = total_loss / total_missing
        print(f"[epoch {e}] 平均损失: {avg_loss:.6f}")

        if e == 1 or e % epoch_per_step == 0:
            model_path = os.path.join(model_dir, f"{model_name}_epoch_{e}.pt")
            save_admm_net(model, model_path)

    return model


In [592]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AdmmNet(
    iter_count = 6,
    conv_channel_cnt = 8,
    init_tau_x = 0.1,
    init_tau_z = 0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)

In [593]:
epoch = 1
epoch_per_step = 10
epoch_count = 100
model_dir = os.path.join(project_dir, "model")
model_name = "admm_net"

In [594]:
# train_admm_net(device, model, training_dataloader, optimizer, epoch, epoch_per_step, epoch_count, model_dir, model_name)

In [595]:
from typing import Dict, Tuple

@torch.no_grad()
def test_admm_net(
    device: torch.device,
    config: dict,
    model_path: str,
    dataloader,
    means: np.ndarray,
    stds: np.ndarray
) -> dict:
    """
    评估 ADMM-Net 重建质量，使用原始尺度计算 RE（包括全体和缺失位置）。
    归一化使用外部传入的 means/stds，适配你已有的预处理流程。

    Args:
        device: 当前设备
        config: 模型配置字典
        model_path: 训练好的权重文件路径
        dataloader: test_loader
        means: 原始数据的每列均值（np.ndarray）
        stds: 原始数据的每列标准差（np.ndarray）

    Returns:
        dict with keys 'RE_all' and 'RE_missing'
    """
    model = load_admm_net(config, model_path).to(device)
    model.eval()

    means = torch.tensor(means, dtype=torch.float32, device=device).view(1, 1, 1, -1)
    stds = torch.tensor(stds, dtype=torch.float32, device=device).view(1, 1, 1, -1)

    num_sq_all = 0.0
    den_sq_all = 0.0
    num_sq_miss = 0.0
    den_sq_miss = 0.0

    for x, m, y in dataloader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        # 推理
        p, _, _ = model(x, torch.zeros_like(x))

        # 反归一化
        p = p * stds + means
        y = y * stds + means

        diff = p - y
        num_sq_all += torch.sum(diff * diff).item()
        den_sq_all += torch.sum(y * y).item()

        missing = ~m if m.dtype == torch.bool else (m == 0)
        if missing.any():
            dm = diff[missing]
            ym = y[missing]
            num_sq_miss += torch.sum(dm * dm).item()
            den_sq_miss += torch.sum(ym * ym).item()

    RE_all = (num_sq_all / den_sq_all) ** 0.5 if den_sq_all > 0 else float("nan")
    RE_missing = (num_sq_miss / den_sq_miss) ** 0.5 if den_sq_miss > 0 else float("nan")

    print(f"[TEST] RE_all (论文口径): {RE_all:.6f}")
    print(f"[TEST] RE_missing(仅缺失处): {RE_missing:.6f}")
    return {"RE_all": RE_all, "RE_missing": RE_missing}



In [596]:
# test_set_data = pd.read_csv(test_set_path)[weather_cols].values
# test_set_normalized_data, test_means, test_stds = to_normalized_data(test_set_data)

# test_set_mask = np.load(test_set_mask_path)
# test_dataset = WeatherDataset(test_set_normalized_data, test_set_mask)
# test_dataloader = DataLoader(test_dataset, batch_size = 512, shuffle = True)

# config = {
#     "iter_count": 6,
#     "conv_channel_count": 8,
#     "init_tau_x": 0.1,
#     "init_tau_z": 0.1
# }

# model_path = os.path.join(model_dir, "admm_net_epoch_0.pt")

# metrics = test_admm_net(
#     device=device,
#     config=config,
#     model_path=model_path,
#     dataloader=test_dataloader,
#     means=test_means,
#     stds=test_stds
# )

# DDQN

雷达模式(radar mode)
$$
a_{k}^{r} \\
k \in \{ 1, 2 \} \\
\text{short-range detection mode}: a_{1}^{r} \\
\text{long-range detection mode}: a_{2}^{r} \\

$$

通信模式(communication mode)
$$
a_{l}^{d}\\
d \in \{ \text{CAM}, \text{DENM} \} \\
l \in \{ 1, 2, 3, 4 \} \\
$$

In [597]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim = 2 + 8, hidden_dims = [128, 128]):
        super().__init__()
        layers = []
        in_features = state_dim
        for h_features in hidden_dims:
            layers.append(nn.Linear(in_features, h_features))
            layers.append(nn.ReLU())
            in_features = h_features

        layers.append(nn.Linear(in_features, action_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, state):
        return self.network(state)

In [598]:
def save_q_net(model, path):
    torch.save(model.state_dict(), path)
    print(f"已保存 Q-Network 模型至: {path}")

def load_q_net(config, path):
    state_dim = config["state_dim"]
    action_dim = config["action_dim"]
    hidden_dims = config["hidden_dims"]

    model = QNetwork(state_dim, action_dim, hidden_dims)
    state_dict = torch.load(path, map_location = torch.device("cpu"))
    model.load_state_dict(state_dict)

    print(f"已加载 Q-Network 模型自: {path}")

    return model

$$
\begin{align*}

\text{Q-VALUE} &= Q(s, a; \theta) \\

a &= \operatorname*{argmax}_{a_j \in \mathcal{A}} Q(s,a_j;\theta) \\

\end{align*}
$$


In [599]:
@torch.no_grad()
def q_vals_for_states(network, states):
    return network(states)

@torch.no_grad()
def q_vals_for_actions(q_vals, actions):
    return q_vals.gather(1, actions)

@torch.no_grad()
def actions_for_q_vals(q_vals):
    return q_vals.argmax(dim = 1, keepdim = True)

## State Transition Function

### Based on the canonical definition of Markov decision processes:
$$
\begin{align*}
\mathcal{T} &= P(s' | s, a) \\
\end{align*}
$$

### Paper:
$$
\begin{align*}
\mathcal{T} &= P(s' | p_{u}, s, n, a) \\
\end{align*}
$$

In [600]:
class StateTransitionModel(nn.Module):
    def __init__(self, state_dim, action_dim = 2 + 8, hidden_dims = [32]):
        super().__init__()
        layers = []
        in_features = state_dim + action_dim
        for h_features in hidden_dims:
            layers.append(nn.Linear(in_features, h_features))
            layers.append(nn.ReLU())
            in_features = h_features

        layers.append(nn.Linear(in_features, state_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, state, action_onehot):
        return self.network(torch.cat([state, action_onehot], dim = -1))

## Reward

$$
r_{com}(t) = 
\begin{cases}

\alpha_{1}l & a = a_{l}^{d}, m = 1, X_{t} = 0 \\

-\alpha_{2}l & a = a_{l}^{d}, X_{t} = 1 \\

0 & a = a_{k}^{r} \\

\end{cases}
$$


In [601]:
# if random.random() < p_u:
#     X_t = 1
# else:
#     X_t = 0

In [602]:
def com_reward(alpha1, alpha2, l, is_comm_action, is_cam_message, is_accident):
    is_comm_action = is_comm_action.float()
    is_cam_message = is_cam_message.float()
    is_accident    = is_accident.float()

    reward_cam_normal = alpha1 * l * is_comm_action * is_cam_message * (1 - is_accident)
    reward_comm_accident = -alpha2 * l * is_comm_action * is_accident

    return reward_cam_normal + reward_comm_accident


$$
r_{rad}(t) = 
\begin{cases}

-\beta_{1}(1 - p_{u}(t)) & a = a_{k}^{r}, X_{t} = 0 \\

\beta_{2}(1 - p_{k}(t)) & a = a_{k}^{r}, X_{t} = 1 \\

\end{cases}
$$


In [603]:
# env = (w, r, n, v, m_ack)
def compute_p_u(env, v_max=40.0):
    """
    env: tensor of shape [batch_size, 5]
         columns = (w, r, n, v, m_ack)

    returns:
        p_u: tensor of shape [batch_size]
    """
    # unpack columns
    w = env[:, 0]
    r = env[:, 1]
    n = env[:, 2]
    v = env[:, 3]
    m_ack = env[:, 4]

    # ----- normalize hazard variables to [0,1] -----
    # w, r, n ∈ {1..10}
    w_norm = (w - 1) / 9
    r_norm = (r - 1) / 9
    n_norm = (n - 1) / 9

    # v normalization
    v_norm = torch.clamp(v / v_max, 0.0, 1.0)

    # m_ack ∈ {1(success), 2(fail)}  →  fail = 1.0, success = 0.0
    ack_norm = torch.where(m_ack == 2, 1.0, 0.0)

    # ----- weights (tunable) -----
    w_w = 0.25
    r_w = 0.25
    n_w = 0.25
    v_w = 0.20
    ack_w = 0.05

    # ----- weighted sum -----
    raw = (
        w_norm * w_w +
        r_norm * r_w +
        n_norm * n_w +
        v_norm * v_w +
        ack_norm * ack_w
    )

    # ----- squash to probability [0,1] -----
    # sigmoid((raw - 0.5) * 5)
    p_u = torch.sigmoid((raw - 0.5) * 5.0)

    return p_u



In [604]:
# p_u, p_k, delta_i

In [605]:
def rad_reward(beta1, p_u, beta2, p_k, is_radar_action, is_accident):
    is_radar_action = is_radar_action.float()
    is_accident = is_accident.float()

    reward_no_accident = -beta1 * (1.0 - p_u) * is_radar_action * (1 - is_accident)
    reward_accident = beta2 * (1.0 - p_k) * is_radar_action * is_accident

    return reward_no_accident + reward_accident

$$
r_{age}(t) =
\begin{cases}

q_{\max} - \delta_{i} & a = a_{l}^{\text{DENM}}, \\

0 & a \neq a_{l}^{\text{DENM}} \\

\end{cases}
$$


In [606]:
def age_reward(q_max, delta_i, is_denm_action):
    is_denm_action = is_denm_action.float()
    return (q_max - delta_i) * is_denm_action

$$
\begin{align*}
R(t) &= \alpha r_{com}(t) + \beta r_{rad}(t) + \lambda r_{age}(t) \\
\end{align*}
$$

In [607]:
def reward(alpha, r_com, beta, r_rad, param_lambda, r_age):
    return alpha * r_com + beta * r_rad + param_lambda * r_age 

In [608]:
# class Environment:
#     def __init__(self):
        

$$
\begin{align*}
y &= r + \gamma (1 - done)\, Q'(s', a; \theta') \\
\end{align*}
$$

In [609]:
@torch.no_grad()
def compute_targets(rewards, have_no_next_action, target_q_vals, gamma):
    return rewards + gamma * (1 - have_no_next_action) * target_q_vals

$$
\begin{align*}
\mathcal{L}(\theta) &= \frac{1}{2}\left( Q(s, a; \theta) - y \right)^{2}
\end{align*}
$$

In [610]:
def compute_loss(online_q_vals, targets):
    return (0.5 * (online_q_vals - targets).pow(2)).mean()

In [611]:
class DDQNUpdater:
    def __init__(self, online_q_network, target_q_network, optimizer):
        self.online_q_network = online_q_network
        self.target_q_network = target_q_network
        self.optimizer = optimizer

    def update_online_q_net(self, loss):
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target_q_net(self):
        self.target_q_network.load_state_dict(
            self.online_q_network.state_dict()
        )

In [612]:
from collections import deque

In [613]:
class ReplayDeque:
    def __init__(self, maxlen):
        self.deque = deque(maxlen = maxlen)

    def append(self, state, action, next_state, reward, has_no_next_action):
        self.deque.append(
            (
                np.array(state, copy = False),
                action,
                np.array(next_state, copy = False),
                reward,
                has_no_next_action
            )
        )

    def sample(self, batch_size):
        states, actions, next_states, rewards, have_no_next_action = zip(*random.sample(self.deque, batch_size))

        return (
            np.stack(states),
            np.array(actions),
            np.stack(next_states),
            np.array(rewards, dtype = np.float32),
            np.array(have_no_next_action, dtype = np.float32)
        )

    def __len__(self):
        return len(self.deque)

In [614]:
class DDQNTrainer:
    def __init__(self, online_q_network, target_q_network, updater, replay_buffer, gamma, batch_size):
        self.online_q_network = online_q_network
        self.target_q_network = target_q_network
        self.updater = updater
        self.replay_buffer = replay_buffer
        self.gamma = gamma
        self.batch_size = batch_size

    def train_step(self):
        # 经验池不够，跳过（你之前说暂不判断，这里给出标准写法，你可删）
        if len(self.replay_buffer) < self.batch_size:
            return None

        # 采样 batch
        states, actions, next_states, rewards, dones = self.replay_buffer.sample(self.batch_size)

        # numpy → tensor
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long).unsqueeze(1)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        rewards = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1)
        dones = torch.tensor(dones, dtype=torch.float32).unsqueeze(1)

        # Q(s, a; θ)
        q_vals = q_vals_for_states(self.online_q_network, states)
        q_vals_for_taken_actions = q_vals_for_actions(q_vals, actions)

        # Double-DQN target
        # Step 1: online network 做 argmax
        next_q_vals_online = q_vals_for_states(self.online_q_network, next_states)
        next_actions = actions_for_q_vals(next_q_vals_online)

        # Step 2: target network 取 Q'(s', argmax)
        next_q_vals_target = q_vals_for_states(self.target_q_network, next_states)
        target_q_vals = q_vals_for_actions(next_q_vals_target, next_actions)

        # 计算最终 target y
        targets = compute_targets(rewards, dones, target_q_vals, self.gamma)

        # 损失
        loss = compute_loss(q_vals_for_taken_actions, targets)

        # 更新 online
        self.updater.update_online_q_net(loss)

        return loss.item()


# 训练